# 도로 재비산먼지 데이터 전처리 및 예측 모델

**데이터 특성**
- 측정 방식: 측정차량이 도로를 주행하며 비정기 측정 (연속 모니터링 아님)
- 측정 건수: 8,280건 (2023~2025, 347개 날짜)
- 구별 연간 ~110건 → 대기 PM10(시간 연속)과 근본적으로 다른 구조
- 좌표 없음 → 주소 텍스트 기반

**처리 목표**
1. 원시 데이터 로드 및 정제
2. 피처 엔지니어링 (시간/도로/기상)
3. 대기 PM10과의 연결
4. 예측 모델 (RF/XGBoost)
5. 격자 단위 확장 (geocoding → grid 매핑)

In [ ]:
import pandas as pd
import numpy as np
import os
import openpyxl
import warnings
warnings.filterwarnings('ignore')

road_pm_23 = "/home/data/youngwoong/ST-GNN_Dataset/0-2. 도로 미세먼지/서울 비재산먼지_23년도.xlsx"
road_pm_24 = "/home/data/youngwoong/ST-GNN_Dataset/0-2. 도로 미세먼지/서울 비재산먼지_24년도.xlsx"
road_pm_25 = "/home/data/youngwoong/ST-GNN_Dataset/0-2. 도로 미세먼지/서울 비재산먼지_25년도.xlsx"
SAVE_DIR   = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/도로 재비산먼지"
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. 원시 데이터 로드

In [ ]:
def load_road_pm(path):
    wb   = openpyxl.load_workbook(path, data_only=True)
    rows = list(wb.active.iter_rows(values_only=True))
    wb.close()
    return pd.DataFrame(rows[1:], columns=rows[0])

dfs = [load_road_pm(p) for p in [road_pm_23, road_pm_24, road_pm_25]]
df_raw = pd.concat(dfs, ignore_index=True)

print(f'원시 데이터 shape: {df_raw.shape}')
print(f'컬럼: {df_raw.columns.tolist()}')
df_raw.head(3)

## 2. 기본 정제

In [ ]:
df = df_raw.copy()

# 타입 변환
df['측정일자']  = pd.to_datetime(df['측정일자'])
df['재비산먼지'] = pd.to_numeric(df['재비산먼지 평균농도(㎍/㎥)'], errors='coerce')
df['기온']      = pd.to_numeric(df['기온(℃)'],  errors='coerce')
df['습도']      = pd.to_numeric(df['습도(%)'],   errors='coerce')
df['측정거리']  = pd.to_numeric(df['측정거리(km)'], errors='coerce')
df['지역명']    = df['지역명'].str.strip()
df['도로명']    = df['도로명'].str.strip()

# 결측치 제거
before = len(df)
df = df.dropna(subset=['재비산먼지', '측정일자', '지역명', '도로명'])
print(f'결측치 제거: {before} → {len(df)}건 ({before-len(df)}건 제거)')

# 음수 제거
df = df[df['재비산먼지'] >= 0]
print(f'음수 제거 후: {len(df)}건')
print(f'기간: {df["측정일자"].min().date()} ~ {df["측정일자"].max().date()}')

# 이상치 확인
print(f'\n재비산먼지 통계:')
print(df['재비산먼지'].describe())

## 3. 탐색적 분석

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
matplotlib.rc('font', family='NanumGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('도로 재비산먼지 탐색적 분석', fontsize=14, fontweight='bold')

# 1. 분포
ax = axes[0,0]
ax.hist(df['재비산먼지'].clip(0,150), bins=50, color='#3498db', edgecolor='white')
ax.set_xlabel('재비산먼지 (μg/m³)'); ax.set_ylabel('건수')
ax.set_title('농도 분포 (0~150 클리핑)')
ax.grid(alpha=0.3)

# 2. 월별 평균
ax = axes[0,1]
monthly = df.groupby(df['측정일자'].dt.month)['재비산먼지'].agg(['mean','count'])
ax.bar(monthly.index, monthly['mean'], color='#e74c3c', alpha=0.8)
ax.set_xlabel('월'); ax.set_ylabel('평균 재비산먼지 (μg/m³)')
ax.set_title('월별 평균 농도')
ax.set_xticks(range(1,13))
ax.grid(axis='y', alpha=0.3)

# 3. 구별 평균
ax = axes[0,2]
gu_mean = df.groupby('지역명')['재비산먼지'].mean().sort_values(ascending=True)
ax.barh(gu_mean.index, gu_mean.values, color='#2ecc71', alpha=0.8)
ax.set_xlabel('평균 재비산먼지 (μg/m³)')
ax.set_title('구별 평균 농도')
ax.tick_params(labelsize=7)

# 4. 기온 vs 재비산먼지
ax = axes[1,0]
ax.scatter(df['기온'], df['재비산먼지'].clip(0,200), alpha=0.2, s=5, color='#9b59b6')
ax.set_xlabel('기온 (℃)'); ax.set_ylabel('재비산먼지 (μg/m³)')
ax.set_title('기온 vs 재비산먼지')
ax.grid(alpha=0.3)

# 5. 습도 vs 재비산먼지
ax = axes[1,1]
ax.scatter(df['습도'], df['재비산먼지'].clip(0,200), alpha=0.2, s=5, color='#e67e22')
ax.set_xlabel('습도 (%)'); ax.set_ylabel('재비산먼지 (μg/m³)')
ax.set_title('습도 vs 재비산먼지')
ax.grid(alpha=0.3)

# 6. 연도별 측정 건수
ax = axes[1,2]
yr_cnt = df.groupby(df['측정일자'].dt.year).size()
ax.bar(yr_cnt.index, yr_cnt.values, color='#1abc9c', alpha=0.8)
ax.set_xlabel('연도'); ax.set_ylabel('측정 건수')
ax.set_title('연도별 측정 건수')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 피처 엔지니어링

In [ ]:
# ── 시간 피처 ─────────────────────────────────────────────────────────────
df['year']      = df['측정일자'].dt.year
df['month']     = df['측정일자'].dt.month
df['weekday']   = df['측정일자'].dt.weekday
df['is_weekend'] = (df['weekday'] >= 5).astype(int)

# 계절 (0=겨울, 1=봄, 2=여름, 3=가을)
df['season'] = df['month'].map({
    12:0, 1:0, 2:0,
    3:1,  4:1, 5:1,
    6:2,  7:2, 8:2,
    9:3, 10:3, 11:3
})

# 순환 인코딩
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# ── 도로/측정 피처 ────────────────────────────────────────────────────────
df['도로길이'] = df['측정거리'].fillna(df['측정거리'].median())

# 구 → 인코딩
from sklearn.preprocessing import LabelEncoder
le_gu   = LabelEncoder().fit(df['지역명'])
le_road = LabelEncoder().fit(df['도로명'])
df['gu_enc']   = le_gu.transform(df['지역명'])
df['road_enc'] = le_road.transform(df['도로명'])

# ── 기상 피처 보완 ────────────────────────────────────────────────────────
df['기온']  = df['기온'].fillna(df.groupby('month')['기온'].transform('median'))
df['습도']  = df['습도'].fillna(df.groupby('month')['습도'].transform('median'))

# 습도 구간 (낮을수록 재비산 높음)
df['is_dry'] = (df['습도'] < 40).astype(int)

print('피처 엔지니어링 완료')
print(f'총 피처: {df.shape[1]}개 컬럼')
df[['측정일자','지역명','도로명','재비산먼지','기온','습도','season','month_sin','도로길이']].head()

## 5. 대기 PM10 연결 (같은 날, 같은 구 기준)

In [ ]:
# ── 대기 PM10 데이터 로드 (ST-GNN 학습 기반) ──────────────────────────────
import sys
HIDDEN_DIR = '/workspace/ST-GNN Modeling/HiddenExtension_V1/data/hidden_vectors'
GDIR       = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본'

stations    = np.load(f'{HIDDEN_DIR}/stations.npy')       # (40,) 측정소명
coords_sta  = np.load(f'{HIDDEN_DIR}/coords.npy')         # (40, 2)

# 대기 PM10 관측값 - 시나리오 CSV에서 로드
SCENARIO_DIR = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/ST-GNN/feature_scenarios'
pm_csv_path  = os.path.join(SCENARIO_DIR, 'S3_transport_pm10_pollutants.csv')

if os.path.exists(pm_csv_path):
    pm_obs = pd.read_csv(pm_csv_path)
    pm_obs['time'] = pd.to_datetime(pm_obs['time'])
    print(f'대기 PM10 데이터: {pm_obs.shape}')
    print(f'컬럼: {pm_obs.columns.tolist()[:8]}')
    
    # 일별 구별 PM10 평균 계산
    # 측정소와 구 매핑
    station_gu_map = {
        '강남구': '강남구', '강동구': '강동구', '강북구': '강북구', '강서구': '강서구',
        '관악구': '관악구', '광진구': '광진구', '구로구': '구로구', '금천구': '금천구',
        '노원구': '노원구', '도봉구': '도봉구', '동대문구': '동대문구', '동작구': '동작구',
        '마포구': '마포구', '서대문구': '서대문구', '서초구': '서초구', '성동구': '성동구',
        '성북구': '성북구', '송파구': '송파구', '양천구': '양천구', '영등포구': '영등포구',
        '용산구': '용산구', '은평구': '은평구', '종로구': '종로구', '중구': '중구', '중랑구': '중랑구'
    }
    
    # 구별 대기 PM10 일평균
    pm_obs['date'] = pm_obs['time'].dt.date
    pm_obs['gu']   = pm_obs['측정소명'].map(lambda x: x.split('구')[0]+'구' if '구' in str(x) else x)
    
    daily_gu_pm = pm_obs.groupby(['date','gu'])['PM10'].mean().reset_index()
    daily_gu_pm.columns = ['date','지역명','ambient_pm10_daily']
    daily_gu_pm['date'] = pd.to_datetime(daily_gu_pm['date'])
    
    # 도로 데이터와 merge
    df['date'] = df['측정일자'].dt.normalize()
    df = df.merge(daily_gu_pm, on=['date','지역명'], how='left')
    print(f'대기 PM10 연결 완료: {df["ambient_pm10_daily"].notna().sum()}/{len(df)}건 매칭')
else:
    print('대기 PM10 CSV 파일 없음 - ambient_pm10_daily 컬럼 생략')
    df['ambient_pm10_daily'] = np.nan

## 6. 재비산먼지 예측 모델 (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# ── 피처 선택 ─────────────────────────────────────────────────────────────
BASE_FEATURES = [
    '기온', '습도', 'is_dry',
    'month_sin', 'month_cos',
    'season', 'is_weekend', 'weekday',
    '도로길이', 'gu_enc',
]

# 대기 PM10이 있으면 추가
if df['ambient_pm10_daily'].notna().sum() > 100:
    FEATURES = BASE_FEATURES + ['ambient_pm10_daily']
    df_model = df[FEATURES + ['재비산먼지']].dropna()
else:
    FEATURES = BASE_FEATURES
    df_model = df[FEATURES + ['재비산먼지']].dropna()

X = df_model[FEATURES].values
y = df_model['재비산먼지'].values

print(f'학습 데이터: {len(X)}건  피처 수: {len(FEATURES)}')
print(f'피처: {FEATURES}')

# ── Train/Test 분할 (시간 순서 기반) ─────────────────────────────────────
# 2023~2024 → train, 2025 → test
df_model_idx = df[FEATURES + ['재비산먼지','year']].dropna()
train_mask = df_model_idx['year'] < 2025
X_tr = df_model_idx.loc[train_mask, FEATURES].values
y_tr = df_model_idx.loc[train_mask, '재비산먼지'].values
X_te = df_model_idx.loc[~train_mask, FEATURES].values
y_te = df_model_idx.loc[~train_mask, '재비산먼지'].values

print(f'\nTrain: {len(X_tr)}건 (2023~2024)')
print(f'Test : {len(X_te)}건 (2025)')

In [ ]:
# ── 모델 학습 ─────────────────────────────────────────────────────────────
models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=200, max_depth=10,
        min_samples_leaf=5, n_jobs=-1, random_state=42
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=4,
        learning_rate=0.05, random_state=42
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    results[name] = {
        'model':     model,
        'train_mae': mean_absolute_error(y_tr, pred_tr),
        'test_mae':  mean_absolute_error(y_te, pred_te),
        'test_r2':   r2_score(y_te, pred_te),
        'pred_te':   pred_te,
    }

# Python 3.6 호환 출력
header = '{:<20} {:>10} {:>10} {:>8}'.format('모델', 'train_MAE', 'test_MAE', 'test_R2')
print(header)
print('-' * 55)
for name, r in results.items():
    row = '{:<20} {:>10.2f} {:>10.2f} {:>8.4f}'.format(
        name, r['train_mae'], r['test_mae'], r['test_r2'])
    print(row)

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────
best_model_name = min(results, key=lambda k: results[k]['test_mae'])
best_model = results[best_model_name]['model']

fi = pd.DataFrame({
    'feature': FEATURES,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance
ax = axes[0]
ax.barh(fi['feature'], fi['importance'], color='#3498db', alpha=0.8)
ax.set_title(f'Feature Importance ({best_model_name})', fontsize=12)
ax.set_xlabel('Importance')
ax.grid(axis='x', alpha=0.3)

# 예측 vs 실측
ax = axes[1]
pred_te = results[best_model_name]['pred_te']
ax.scatter(y_te, pred_te, alpha=0.3, s=10, color='#e74c3c')
lim = max(y_te.max(), pred_te.max())
ax.plot([0, lim], [0, lim], 'k--', linewidth=1)
ax.set_xlabel('실측 재비산먼지 (μg/m³)')
ax.set_ylabel('예측 재비산먼지 (μg/m³)')
ax.set_title(f'예측 vs 실측  MAE={results[best_model_name]["test_mae"]:.2f}')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 전처리된 데이터 저장

In [ ]:
# ── 정제 데이터 저장 ──────────────────────────────────────────────────────
save_cols = [
    '측정일자', '지역명', '도로명', '시작점', '종점', '측정거리',
    '기온', '습도', '재비산먼지', '상태', 'season', 'month', 'weekday', 'is_weekend'
]
if 'ambient_pm10_daily' in df.columns:
    save_cols.append('ambient_pm10_daily')

df_save = df[save_cols].copy()
out_csv = os.path.join(SAVE_DIR, 'road_resusp_pm_cleaned.csv')
df_save.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'정제 데이터 저장: {out_csv}')
print(f'shape: {df_save.shape}')

# ── 모델 저장 ─────────────────────────────────────────────────────────────
import joblib
model_path = os.path.join(SAVE_DIR, f'model_{best_model_name.lower()}.pkl')
joblib.dump(best_model, model_path)
print(f'모델 저장: {model_path}')

# ── 피처 이름 저장 ────────────────────────────────────────────────────────
import json
meta = {
    'features': FEATURES,
    'best_model': best_model_name,
    'test_mae': results[best_model_name]['test_mae'],
    'test_r2':  results[best_model_name]['test_r2'],
    'train_period': '2023~2024',
    'test_period':  '2025',
    'n_train': int(len(X_tr)),
    'n_test':  int(len(X_te)),
}
with open(os.path.join(SAVE_DIR, 'model_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('메타 저장 완료')
print()
print('=== 최종 결과 ===')
print(f'  모델: {best_model_name}')
print(f'  Test MAE: {meta["test_mae"]:.2f} μg/m³')
print(f'  Test R²:  {meta["test_r2"]:.4f}')

## 8. 한계 및 다음 단계

### 현재 모델의 한계

```
1. 비정기 측정: 연속 시계열이 아닌 현장조사 데이터
   → 특정 날짜 예측 불가 (측정 없는 날이 대부분)

2. 좌표 없음: 주소 텍스트만 존재
   → 격자 매핑을 위해 geocoding 필요

3. 교통량 미연결: 재비산먼지의 핵심 동인인 교통량 데이터 미사용
   → 교통량 데이터(2-2. 도로 및 교통 데이터)와 연결 필요
```

### 다음 단계 (격자 확장)

```
Step 1: Geocoding
  시작점/종점 주소 → 위경도 좌표
  도로 선분(LineString) 생성

Step 2: 격자 매핑
  각 250m 격자와 교차하는 도로 선분 연결
  → 격자별 "포함 도로" 목록 생성

Step 3: 격자 재비산먼지 추정
  격자 내 도로들의 예측값 가중 평균
  (도로 길이 가중치 or 교통량 가중치)

Step 4: 대기 PM10과의 결합
  ambient PM10 (V5-base) + road resusp PM
  → 도로 인근 격자에서 PM10 보정
```